# 02 Residual Stream / Logit Lens / Activation Patching

## このノートブックの目的

Qwen3-4B の内部計算を 3 つの視点から観察する。

| 概念 | 概要 |
|------|------|
| **Residual Stream** | 各 Transformer layer の入出力となる hidden state のベクトル系列 |
| **Logit Lens** | 各 layer 後の hidden state を lm_head に通して「その時点でのモデルの予測」を読む |
| **Activation Patching** | clean run の hidden state を corrupt run に注入し、各 layer の因果的寄与を測る |

### 実験設定

| | プロンプト | 期待する次トークン |
|---|---|---|
| **clean** | `"The capital of Japan is"` | ` Tokyo` |
| **corrupt** | `"The capital of France is"` | ` Paris` |

clean run の hidden state を layer k で corrupt run に注入したとき、出力がどう変化するかを計測する。


## 1. 環境セットアップ

In [ ]:
%matplotlib inline
import json
from pathlib import Path
import torch
import inspect
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_ID = "Qwen/Qwen3-4B"

# Jupyter はこのノートのあるフォルダから起動してください
outputs_dir = Path("../outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)
print(f"model_id : {MODEL_ID}")
print(f"outputs  : {outputs_dir.name}/")

# デバイス選択
if torch.cuda.is_available():
    device = "cuda"
    torch_dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    torch_dtype = torch.float16
else:
    device = "cpu"
    torch_dtype = torch.float32
print(f"device   : {device}")
print(f"dtype    : {torch_dtype}")


## 2. モデルとトークナイザーの読み込み

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedModel

# トークナイザーの読み込み
print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"  vocab_size = {tokenizer.vocab_size}")

# モデルの読み込み
print("Loading model ...")
model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,
    attn_implementation="eager",
)
model.to(device)  # GPU/MPS/CPU に転送  # pyright: ignore[reportArgumentType]
model.eval() # 推論専用の動作に設定（Dropout オフなど）
torch.set_grad_enabled(False)  # 計算グラフ構築オフ
print(f"  device     : {next(model.parameters()).device}")
print(f"  dtype      : {next(model.parameters()).dtype}")

K: int = model.config.num_hidden_layers
hidden_size: int = model.config.hidden_size
print(f"  layers K   : {K}")
print(f"  hidden_size: {hidden_size}")


### ヘルパー関数 (tokenizer使用)

In [ ]:
# トークンテーブルを表示
def show_token_table(text: str) -> pd.DataFrame:
    """テキストをトークナイズして位置・ID・piece の表を返す。"""

    # add_special_tokens=False: BOS などを付加せず、テキスト本体だけをトークン化
    ids = tokenizer.encode(text, add_special_tokens=False)

    rows = []
    for pos, tid in enumerate(ids):
        # convert_ids_to_tokens: トークナイザー内部の raw 表現（Ġ 付きなど）
        piece = tokenizer.convert_ids_to_tokens(tid)
        # decode: token ID を人間が読めるテキストに変換
        decoded = tokenizer.decode([tid])
        rows.append({
            "pos":      pos,
            "token_id": tid,
            "piece":    piece,   
            "decoded":  repr(decoded), # 'capital' のように表示
        })
    return pd.DataFrame(rows).set_index("pos")

# 確率上位 k トークンの表を返す
def topk_table(logits: torch.Tensor, k: int = 10) -> pd.DataFrame:
    """logits [vocab] から確率上位 k トークンの DataFrame を返す。"""

    # logits をソフトマックスで確率に変換（float32 で計算して精度を確保）
    probs = torch.softmax(logits.float(), dim=-1)

    # 確率上位 k 個の値とインデックスを取得
    top_vals, top_ids = torch.topk(probs, k)

    rows = []
    for rank, (tid, prob) in enumerate(zip(top_ids.tolist(), top_vals.tolist()), start=1):
        decoded = tokenizer.decode([tid])   # token ID → テキスト
        rows.append({
            "rank":     rank,
            "token_id": tid,
            "decoded":  repr(decoded),    # 先頭スペースなどを repr で可視化
            "logit":    logits[tid].item(),
            "prob":     prob,
        })
    return pd.DataFrame(rows).set_index("rank")

## 3. トークンテーブル

In [ ]:
# プロンプトと答え
CLEAN_PROMPT   = "The capital of Japan is"
CORRUPT_PROMPT = "The capital of France is"
CLEAN_ANSWER   = " Tokyo"
CORRUPT_ANSWER = " Paris"

clean_ans_ids   = tokenizer.encode(CLEAN_ANSWER,   add_special_tokens=False)
corrupt_ans_ids = tokenizer.encode(CORRUPT_ANSWER, add_special_tokens=False)

CLEAN_ANS_ID   = clean_ans_ids[0]
CORRUPT_ANS_ID = corrupt_ans_ids[0]
print(f"CLEAN_ANSWER   {CLEAN_ANSWER!r} -> id = {CLEAN_ANS_ID}")
print(f"CORRUPT_ANSWER {CORRUPT_ANSWER!r} -> id = {CORRUPT_ANS_ID}")
if len(clean_ans_ids) != 1 or len(corrupt_ans_ids) != 1:
    print("[warning] answer string is not a single token")


In [ ]:
df_clean_tok = show_token_table(CLEAN_PROMPT)
display(df_clean_tok.style.set_caption("clean prompt token table"))
clean_pos = len(df_clean_tok) - 1

df_corrupt_tok = show_token_table(CORRUPT_PROMPT)
display(df_corrupt_tok.style.set_caption("corrupt prompt token table"))
corrupt_pos = len(df_corrupt_tok) - 1

## 4. Forward pass の計算

In [ ]:
# clean run
clean_inputs = tokenizer(CLEAN_PROMPT, return_tensors="pt").to(device)
clean_outputs = model(
    **clean_inputs,
    output_hidden_states=True,   # 全 layer の hidden state を返す（logit lens に必要）
    output_attentions=False,     # attention weights は不要（メモリ節約）
    use_cache=False,             # KV cache 無効（内部状態観察時は不要）
)

# 残差ストリーム　（全層，トークン列の全体）
# hidden_states: K+1 個のテンソルのタプル, 各テンソルの shape = [1, seq_len, hidden_size]
#   hs[0]   = embed_tokens の出力
#   hs[k]   = layers[k-1] の出力, k=1,2,...,K-1
#   hs[K]   = layers[K-1]の出力に，final normを適用したもの
clean_hs = clean_outputs.hidden_states

# 次トークンの予測分布の logit 値　（トークン列の全体）
# logits: shape [1, seq_len, vocab_size]　（ここではテキスト一つだからバッチサイズは1）
clean_logits = clean_outputs.logits[0, clean_pos, :].float() # clean_posの logit ベクトル
#   shape [vocab_size] の1次元テンソルになる
clean_logits.shape

In [ ]:
# softmax で logit → 確率に変換（全vocabの確率の和 = 1）
#   dim=-1: 最後の次元（vocab_size 方向）に沿って softmax を適用
#   clean_logits は 1次元テンソルなので dim=0 と dim=-1 は等価（多次元では異なる）
clean_probs  = torch.softmax(clean_logits, dim=-1)
print(f"clean run:")
print(f"  top-1   = {repr(tokenizer.decode([clean_logits.argmax().item()]))}")
print(f"  P({CLEAN_ANSWER.strip()}) = {clean_probs[CLEAN_ANS_ID]:.4f}")
print(f"  P({CORRUPT_ANSWER.strip()}) = {clean_probs[CORRUPT_ANS_ID]:.4f}")

In [ ]:
# corrupt run
corrupt_inputs = tokenizer(CORRUPT_PROMPT, return_tensors="pt").to(device)
corrupt_outputs = model(
    **corrupt_inputs,
    output_hidden_states=True,
    output_attentions=False,
    use_cache=False,
)

corrupt_hs     = corrupt_outputs.hidden_states
corrupt_logits = corrupt_outputs.logits[0, corrupt_pos, :].float()
corrupt_probs  = torch.softmax(corrupt_logits, dim=-1)
print(f"corrupt run:")
print(f"  top-1   = {repr(tokenizer.decode([corrupt_logits.argmax().item()]))}")
print(f"  P({CLEAN_ANSWER.strip()}) = {corrupt_probs[CLEAN_ANS_ID]:.4f}")
print(f"  P({CORRUPT_ANSWER.strip()}) = {corrupt_probs[CORRUPT_ANS_ID]:.4f}")


In [ ]:
display(topk_table(clean_logits, k=10).style.set_caption("clean top-10"))
display(topk_table(corrupt_logits, k=10).style.set_caption("corrupt top-10"))

Tokyo/Parisのかわりにin, located, 下線, Kyotoなど，英文として有り得そうな単語も含まれている


## 5. Hidden states と logit lens

In [ ]:
print(f"K = {K}")
print(f"len(clean_hs) = {len(clean_hs)}  (= K+1 = {K}+1)")
print(f"hs[0].shape   = {tuple(clean_hs[0].shape)}  <- embed_tokens 出力")
print(f"hs[1].shape   = {tuple(clean_hs[1].shape)}  <- layer 0 出力")
print(f"hs[K].shape   = {tuple(clean_hs[K].shape)}  <- norm 後 (k=K)")
print()
print("Residual stream のインデックス対応:")
print("  hs[0]   = embed_tokens(input_ids)")
print("  hs[k]   = layers[k-1] の出力  (1 <= k <= K-1)")
print(f"  hs[{K}]  = model.model.norm(layers[{K-1}] の出力)  ← lm_head の直前")

### lm_head を利用して logit lens の定義

In [ ]:
# Logit Lens の計算
def logit_lens(hidden_states, k: int, pos: int, model) -> torch.Tensor:
    """
    残差ストリームの layer k を「その時点でのモデルの予測」として読むために，
    その時点での hidden state を lm_head に通した logits を返す。
    k = Kだけ処理が異なることに注意する。

    hidden_states のインデックス対応:
      hs[0]   = embed_tokens の出力
      hs[k]   = layers[k-1] の出力  （1 <= k <= K-1）
      hs[K]   = model.model.norm 後  （lm_head の直前）

    k < K の場合は norm を通してから lm_head に入力する。
    k = K の場合は hs[K] が既に norm 後なので直接 lm_head に入力する。
    """
    K = len(hidden_states) - 1  # Qwen3-4B では K=36
    hs = hidden_states[k]       # shape: [1, seq, hidden_size]

    if k < K:
        # pos:pos+1 でスライスして [1, 1, hidden_size] にしてから norm を適用
        normed = model.model.norm(hs[:, pos:pos+1, :])  # [1, 1, hidden_size]
        logits = model.lm_head(normed)[:, 0, :]          # [1, vocab]
    else:
        # k == K: hs[K] は既に norm 済み。pos だけ取り出して lm_head へ
        normed = hs[:, pos, :]         # [1, hidden_size]
        logits = model.lm_head(normed) # [1, vocab]

    return logits[0]  # [vocab]


# 全 layer の logit lens を sweep して DataFrame を返す
def logit_lens_sweep(hidden_states, pos: int) -> pd.DataFrame:
    """hidden_states の各層を logit lens で読み、確率等をまとめた DataFrame を返す。"""
    K_loc = len(hidden_states) - 1
    rows = []
    for k in range(K_loc + 1):
        ll_logits    = logit_lens(hidden_states, k, pos, model)
        ll_probs     = torch.softmax(ll_logits.float(), dim=-1)
        top1_id      = int(ll_logits.argmax().item())
        top1_decoded = tokenizer.decode([top1_id])
        site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K_loc else "norm")
        rows.append({
            "k":            k,
            "site":         site,
            "top1_decoded": repr(top1_decoded),
            "p_top1":       ll_probs[top1_id].item(),
            "p_clean":      ll_probs[CLEAN_ANS_ID].item(),
            "p_corrupt":    ll_probs[CORRUPT_ANS_ID].item(),
        })
    return pd.DataFrame(rows)

### lm_head の動作確認

In [ ]:
# hs[K] (norm 後) を lm_head に通した結果と model output logits を比較
ll_logits_K = logit_lens(clean_hs, K, clean_pos, model)
display(topk_table(ll_logits_K, k=5).style.set_caption("logit_lens(k=K)"))
display(topk_table(clean_logits, k=5).style.set_caption("model output logits"))

diff = (ll_logits_K - clean_logits).abs().max().item()
print(f"logit_lens(k=K) vs model logits: max abs diff = {diff:.6f}")
print("  diff ≈ 0 なら logit_lens の実装が正しい")

## 6. Logit Lens — clean run

In [ ]:
# clean run
df_ll_clean = logit_lens_sweep(clean_hs, clean_pos)
display(df_ll_clean.set_index("k").style.set_caption("Logit Lens — clean run").format({"p_top1": "{:.4f}", "p_clean": "{:.4f}", "p_corrupt": "{:.4f}"}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_ll_clean["k"], df_ll_clean["p_clean"], label=f"P({CLEAN_ANSWER})", marker="o", markersize=3)
ax.plot(df_ll_clean["k"], df_ll_clean["p_corrupt"], label=f"P({CORRUPT_ANSWER})", marker="s", markersize=3, linestyle="--")
ax.set_xlabel("layer k  (0=embed, K=norm)")
ax.set_ylabel("probability")
ax.set_title(f"Logit Lens — clean run  ({CLEAN_PROMPT})")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = outputs_dir / "nb02_logit_lens_clean.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")


## 7. Logit Lens 比較 — clean vs corrupt

In [ ]:
# corrupt run
df_ll_corrupt = logit_lens_sweep(corrupt_hs, corrupt_pos)
display(df_ll_corrupt.set_index("k").style.set_caption("Logit Lens — corrupt run").format({"p_top1": "{:.4f}", "p_clean": "{:.4f}", "p_corrupt": "{:.4f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ax = axes[0]
ax.plot(df_ll_clean["k"], df_ll_clean["p_clean"], label=f"P({CLEAN_ANSWER})", marker="o", markersize=3)
ax.plot(df_ll_clean["k"], df_ll_clean["p_corrupt"], label=f"P({CORRUPT_ANSWER})", marker="s", markersize=3, linestyle="--")
ax.set_title(f"Logit Lens — clean  ({CLEAN_PROMPT})")
ax.set_xlabel("layer k")
ax.set_ylabel("probability")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(df_ll_corrupt["k"], df_ll_corrupt["p_clean"], label=f"P({CLEAN_ANSWER})", marker="o", markersize=3)
ax.plot(df_ll_corrupt["k"], df_ll_corrupt["p_corrupt"], label=f"P({CORRUPT_ANSWER})", marker="s", markersize=3, linestyle="--")
ax.set_title(f"Logit Lens — corrupt  ({CORRUPT_PROMPT})")
ax.set_xlabel("layer k")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)

plt.tight_layout()
out = outputs_dir / "nb02_logit_lens_comparison.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")


### logit-difference metric で見る logit lens

確率 plot は後半層で 0/1 に飽和するため、前半・中盤の変化が見えにくい場合がある。
`metric_k = logit_k(" Tokyo") - logit_k(" Paris")` を clean/corrupt で並べると、変化がより見やすくなる。


In [ ]:
# ２単語の出現しやすさの差を計算
def metric(logits: torch.Tensor, clean_id: int, corrupt_id: int) -> float:
    """
    logit(clean_answer) - logit(corrupt_answer) を返す。

    この値が大きいほど clean 側の答えが有利な状態。
    recovery の計算に使う基準指標。
    """
    return (logits[clean_id] - logits[corrupt_id]).item()

In [ ]:
# logit-difference metric plot (clean / corrupt)
ll_metric_clean   = [
    metric(logit_lens(clean_hs,   k, clean_pos,   model), CLEAN_ANS_ID, CORRUPT_ANS_ID)
    for k in range(K + 1)
]
ll_metric_corrupt = [
    metric(logit_lens(corrupt_hs, k, corrupt_pos, model), CLEAN_ANS_ID, CORRUPT_ANS_ID)
    for k in range(K + 1)
]

ks = list(range(K + 1))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ks, ll_metric_clean,   label=f"clean  ({CLEAN_PROMPT})",  marker="o", markersize=3)
ax.plot(ks, ll_metric_corrupt, label=f"corrupt ({CORRUPT_PROMPT})", marker="s", markersize=3, linestyle="--")
ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("layer k  (0=embed, K=norm)")
ax.set_ylabel(f"logit({CLEAN_ANSWER}) - logit({CORRUPT_ANSWER})")
ax.set_title("Logit Lens — metric_k  (logit difference)")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = outputs_dir / "nb02_logit_lens_metric.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")


## 8. Activation Patching の概念

### 概念

Activation Patching（残差ストリームパッチ）の手順：

1. **clean run** を実行 → 各 layer の hidden state `clean_hs[k]` を記録
2. **corrupt run** に hook を設置：layer k の出力を `clean_hs[k][0, clean_pos, :]` で上書き
3. corrupt run を再実行 → 最終 logits を記録
4. **回復率 (recovery)** を計算：

$$\text{recovery}(k) = \frac{\text{metric}(\text{patched}) - \text{metric}(\text{corrupt})}{\text{metric}(\text{clean}) - \text{metric}(\text{corrupt})}$$

metric = `logit( Tokyo) - logit( Paris)`  at last position

- recovery ≈ 0 → layer k のパッチは効果なし
- recovery ≈ 1 → layer k のパッチで clean run に完全回復

### hidden_states インデックスと patch site の対応

| k | patch site | モジュール |
|---|---|---|
| 0 | embed_tokens 出力 | `model.model.embed_tokens` |
| 1 ≤ k ≤ K-1 | layers[k-1] 出力 | `model.model.layers[k-1]` |
| K | norm 出力 | `model.model.norm` |


In [ ]:
def run_patch(k: int) -> torch.Tensor:
    """
    corrupt run の層 k 出力 (corrupt_hs[k][0, corrupt_pos, :]) を
    clean  run の層 k 出力 (clean_hs[k][0,  clean_pos, :])  で置換して
    最終 logits [vocab] を返す。

    Transformers 5.x: Qwen3DecoderLayer.forward() は tensor を直接返す（tuple でない）。
    hook は out.clone() してから置換し、tensor を return する。
    """
    # clean run の層 k における clean_pos の残差ストリームを取り出す
    patch_vec = clean_hs[k][0, clean_pos, :].to(device)

    # patch する対象モジュールを k に応じて選ぶ
    #   k == 0       : embed_tokens（埋め込み層の出力）
    #   1 <= k <= K-1: layers[k-1]（第 k-1 デコーダ層の出力）
    #   k == K       : norm（最終 LayerNorm の出力）
    if k == 0:
        target_module = model.model.embed_tokens
    elif k < K:
        target_module = model.model.layers[k - 1]
    else:
        target_module = model.model.norm

    # forward hook: target_module の出力が計算されるたびに呼ばれる
    #   out[0, corrupt_pos, :] だけを patch_vec で上書きし、残りはそのまま返す
    def hook(module, inp, out):
        out = out.clone()                        # 元テンソルを破壊しないようコピー
        out[0, corrupt_pos, :] = patch_vec       # 指定位置だけ置換
        return out                               # 後続の層はこの値を受け取る

    # hook を登録して corrupt run を再実行し、終了後に必ず hook を解除する
    handle = target_module.register_forward_hook(hook)
    try:
        patched_out = model(
            **corrupt_inputs,
            output_hidden_states=False,
            output_attentions=False,
            use_cache=False,
        )
    finally:
        handle.remove()   # 例外が起きても hook が残り続けないように

    # corrupt_pos における最終 logits を返す
    return patched_out.logits[0, corrupt_pos, :].float()

### hook が挿入されるモジュールのソース確認

> **補足（読み飛ばし可）** : PyTorch の hook 機構と Transformers の実装詳細に踏み込む。activation patching の結果を理解するだけなら読み飛ばしてよい。


`run_patch` は `target_module` に `register_forward_hook` を登録する。
`target_module` の型は k によって異なる。まず各型を確認し、それぞれの `forward` を見る。

In [ ]:
# register_forward_hook が何をするかを確認
# -> self._forward_hooks に hook 関数を登録する
# forward() が呼ばれるたびに _forward_hooks の中身が順番に実行される
import torch.nn as nn
print(inspect.getsource(nn.Module.register_forward_hook))

In [ ]:
# run_patch で使う3種類の target_module の型を確認
print("embed_tokens:", type(model.model.embed_tokens))
print("layers[0]   :", type(model.model.layers[0]))
print("norm        :", type(model.model.norm))

In [ ]:
# modelのどこにモジュールがあるか確認
model

In [ ]:
# Qwen3Model.forward — 計算順序を確認する
from transformers.models.qwen3.modeling_qwen3 import Qwen3Model
print(inspect.getsource(Qwen3Model.forward))

#### `Qwen3Model.forward` と hook の挿入場所

`run_patch` の `target_module` は k に応じて以下の3箇所のいずれかになる。

| k | コード上の該当行 | target_module |
|---|---|---|
| `k == 0` | `inputs_embeds = self.embed_tokens(input_ids)` | `model.model.embed_tokens` |
| `1 <= k <= K-1` | `hidden_states = decoder_layer(hidden_states, ...)` (i番目のループ) | `model.model.layers[k-1]` |
| `k == K` | `hidden_states = self.norm(hidden_states)` | `model.model.norm` |

各モジュールの `forward` が `return` した直後に hook が呼ばれ、  
`hidden_states[0, corrupt_pos, :]` だけを `patch_vec` で置換する。

In [ ]:
# decoder layer の forward を確認
# Qwen3DecoderLayer.forward（k=1..K-1 のとき）
from transformers.models.qwen3.modeling_qwen3 import Qwen3DecoderLayer, Qwen3RMSNorm
print(inspect.getsource(Qwen3DecoderLayer.forward))

#### ソースの読み方（Qwen3DecoderLayer.forward）

```
residual = hidden_states
hidden_states = self.input_layernorm(hidden_states)
hidden_states = self.self_attn(...)          # Attention
hidden_states = residual + hidden_states     # 残差結合①

residual = hidden_states
hidden_states = self.post_attention_layernorm(hidden_states)
hidden_states = self.mlp(hidden_states)      # MLP
hidden_states = residual + hidden_states     # 残差結合②

return hidden_states   # ← hook はここで呼ばれる（return の直後）
```

返り値が tuple でなく **tensor 1つ**であることが確認できる。これが `out.clone()` で直接操作できる理由。  
`nn.Embedding.forward` と `Qwen3RMSNorm.forward` も同様に tensor を返す。

In [ ]:
# nn.Embedding.forward（k=0 のとき）
import torch.nn as nn
print(inspect.getsource(nn.Embedding.forward))

In [ ]:
# Qwen3RMSNorm.forward（k=K のとき）
print(inspect.getsource(Qwen3RMSNorm.forward))

## 9. 全 layer スイープ

In [ ]:
# metric のベースライン（recovery 計算の分母）
clean_metric   = metric(clean_logits,   CLEAN_ANS_ID, CORRUPT_ANS_ID)
corrupt_metric = metric(corrupt_logits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
print(f"clean   metric = {clean_metric:.4f}")
print(f"corrupt metric = {corrupt_metric:.4f}")

# patching sweep
sweep_rows = []
print(f"Patching sweep: k = 0 ... {K}")
for k in range(K + 1):
    plogits = run_patch(k)
    pprobs = torch.softmax(plogits, dim=-1)
    pm  = metric(plogits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
    rec = (pm - corrupt_metric) / (clean_metric - corrupt_metric)
    site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K else "norm")
    sweep_rows.append({
        "k": k,
        "site": site,
        "p_clean_patched": pprobs[CLEAN_ANS_ID].item(),
        "p_corrupt_patched": pprobs[CORRUPT_ANS_ID].item(),
        "patched_metric": pm,
        "recovery": rec,
    })
    if k % 5 == 0 or k == K:
        print(f"  k={k:2d} ({site:6s}): recovery={rec:.4f}")

df_sweep = pd.DataFrame(sweep_rows)
out = outputs_dir / "nb02_patching_sweep.csv"
df_sweep.to_csv(out, index=False)
print(f"Saved: {out.name}")

display(df_sweep.set_index("k").style.set_caption("patching sweep").format({"recovery": "{:.4f}", "patched_metric": "{:.4f}", "p_clean_patched": "{:.4f}", "p_corrupt_patched": "{:.4f}"}))

### パッチ注入後の P(answer)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sweep["k"], df_sweep["p_clean_patched"],   label=f"P({CLEAN_ANSWER})  patched", marker="o", markersize=3)
ax.plot(df_sweep["k"], df_sweep["p_corrupt_patched"], label=f"P({CORRUPT_ANSWER}) patched", marker="s", markersize=3, linestyle="--")
ax.set_xlabel("patch site k  (0=embed, K=norm)")
ax.set_ylabel("probability")
ax.set_title(f"Activation Patching — P(answer) after patch\n({CLEAN_PROMPT}  →  {CORRUPT_PROMPT})")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)

# p_clean_patched が p_corrupt_patched を初めて上回る k に赤線を引く
cross = df_sweep[df_sweep["p_clean_patched"] > df_sweep["p_corrupt_patched"]]
if len(cross) > 0:
    k_cross = int(cross["k"].min())  # pyright: ignore[reportArgumentType]
    ax.axvline(k_cross, color="tomato", linestyle=":", linewidth=1.5,
               label=f"k={k_cross}: P({CLEAN_ANSWER.strip()}) > P({CORRUPT_ANSWER.strip()})")
    ax.legend()

plt.tight_layout()
out = outputs_dir / "nb02_patching_probs.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")
if len(cross) > 0:
    print(f"最初に 確率が入れ替わる layer: k={k_cross}")

### Recovery curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sweep["k"], df_sweep["recovery"], marker="o", markersize=4, color="steelblue", label="recovery")
ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("patch site k  (0=embed, K=norm)")
ax.set_ylabel("recovery")
ax.set_title(f"Activation Patching — Recovery by Layer\n({CLEAN_PROMPT}  →  {CORRUPT_PROMPT})")
ax.set_xticks(range(0, K + 1, 4))
ax.set_ylim(-0.1, 1.1)
ax.grid(True, alpha=0.3)

# 最初に recovery >= 0.5 になる k にマーカー
transition = df_sweep.loc[df_sweep["recovery"] >= 0.5, "k"]
if len(transition) > 0:
    k_transition = int(transition.min())
    ax.axvline(k_transition, color="tomato", linestyle=":", linewidth=1.5,
               label=f"k={k_transition}: recovery≥0.5")
    ax.legend()

plt.tight_layout()
out = outputs_dir / "nb02_recovery_curve.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")
if len(transition) > 0:
    print(f"最初に recovery≥0.5 になる layer: k={k_transition}")


### 補足：Logit Lens と Activation Patching の比較

logit-difference metric で見ると、Logit Lens でも Activation Patching でも k=25 付近が転換点として現れる。
両手法は異なる操作だが、この題材では**同じ層を重要と示している**。

| 操作 | 内容 |
|------|------|
| **Logit Lens** | 層 k の残差ストリームを**その場で** lm_head（+norm）に通して読む |
| **Activation Patching** | 層 k の残差ストリームを差し替えたあと、**残りの層を通常通り forward** させる |

ただし、Activation Patching の recovery curve の方が変化がクリアに見える。
Logit Lens は「その時点の表現を直接読む」ため、後続の層による変換が反映されない。
Activation Patching は残りの層を通してから評価するため、残差ストリームの「影響力」をより直接的に測れる。

なお、確率（softmax）で見ると k=25 での変化は小さく見えるが、これは確率が後半層で急峻に飽和するためであり、
logit-difference metric を使うことで前半・中盤の変化が正確に読み取れる。

## 10. 特定 layer のパッチ後 top-k 比較

In [ ]:
# 特定 layer のパッチ後 logits を計算
plogits_24 = run_patch(24)
plogits_25 = run_patch(25)
plogits_K  = run_patch(K)

labels = [
    ("clean baseline",   clean_logits),
    ("corrupt baseline", corrupt_logits),
    ("patch k=24",       plogits_24),
    ("patch k=25",       plogits_25),
    (f"patch k=K={K}",   plogits_K),
]

for label, lgts in labels:
    rec_val = (metric(lgts, CLEAN_ANS_ID, CORRUPT_ANS_ID) - corrupt_metric) / (clean_metric - corrupt_metric)
    display(topk_table(lgts, k=5).style.set_caption(f"{label}  (recovery={rec_val:.4f})"))

# k=K サニティチェック: recovery=1.000 になるはず
recK = (metric(plogits_K, CLEAN_ANS_ID, CORRUPT_ANS_ID) - corrupt_metric) / (clean_metric - corrupt_metric)
print(f"k=K sanity: recovery = {recK:.6f}  (expected 1.000)")
if abs(recK - 1.0) > 1e-3:
    print("[warning] recovery != 1.000  (hook の実装を確認)")

## 11. まとめ

### 観察結果

| 手法 | 観察したこと |
|------|-------------|
| **Logit Lens** | 前半層（k≤24程度）では top-1 が意味のないトークン、後半層（k≈25以降）から正答 ` Tokyo` が上位に現れる |
| **Activation Patching** | k≤24 では recovery≈0、k=25 付近から急増、k≥34 では recovery≈1.0 |
| **k=K sanity check** | k=K（norm 後）のパッチで recovery=1.000 → hook の実装が正しいことを確認 |

### 解釈（この prompt pair とこの setup における観察）

- この題材・この last-token position・この metric では、**k=25 付近の残差ストリームを差し替えると最終出力が ` Tokyo` 側へ大きく変化した**。
- 少なくとも last-token position の残差ストリームを単独で差し替えるこの実験では、k≤24 の patch は最終出力をほとんど変えなかった。
- ただし、これを「知識が一般に後半層だけにある」と一般化してはいけない。あくまで **この prompt pair・この patching setup における観察**である。
- Logit Lens では ` Tokyo` の確率が明確に上がるのは k≈29 以降だが、Activation Patching では k=25 でも大きな recovery が得られた。これは両手法が**異なる操作**であることを反映している（詳細は Section 9 の注記を参照）。

### 参考文献

- Logit Lens: Nostalgebraist (2020), *Interpreting GPT: the logit lens*
- Activation Patching / Causal Tracing: Meng et al. (2022), ROME
- TransformerLens: Nanda et al. (2022)
